# Search and department access

An engineering employee can read the Germany parental-leave policy. An
outsider cannot. The disciplinary procedure stays with `hr_admin`.

The gateway log line for each search is in Loki. `decision` is `allow` or
`deny`. The line has no snippet.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "gateway_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import gateway_demo
import hr_demo


In [ ]:
employee = gateway_demo.token("employee_eng")
outsider = gateway_demo.token("outsider")
admin = gateway_demo.token("hr_admin")

found = gateway_demo.search(
    employee, "parental leave Germany", request_id="nb-allow"
)
ids = {item["object_id"] for item in found["results"]}
assert "pol-parental-leave-de" in ids
assert found["results"][0]["snippet"].startswith("<<<UNTRUSTED_CONTENT")
assert "Germany" in found["results"][0]["snippet"]

empty = gateway_demo.search(
    outsider, "parental leave Germany", request_id="nb-deny"
)
assert empty["results"] == []

hidden = gateway_demo.search(employee, "disciplinary termination")
assert all(item["object_id"] != "pol-disciplinary-procedure" for item in hidden["results"])
visible = gateway_demo.search(admin, "disciplinary termination")
assert any(item["object_id"] == "pol-disciplinary-procedure" for item in visible["results"])

def allow_line():
    line = gateway_demo.decision_line("nb-allow")
    return line if line and line.get("decision") == "allow" else None

def deny_line():
    line = gateway_demo.decision_line("nb-deny")
    return line if line and line.get("decision") == "deny" else None

allowed = hr_demo.wait_until(allow_line, 30, "allow log")
denied = hr_demo.wait_until(deny_line, 30, "deny log")
assert allowed["count"] >= 1
assert denied["count"] == 0
assert "snippet" not in allowed and "snippet" not in denied

sources = gateway_demo.raw("GET", "/v1/sources").json()
assert any(item["source_id"] == "hr" for item in sources)
print("search ok", sorted(ids))
